In [0]:
UC_CATALOG = "preethiworkspace_7405608993331015"
UC_SCHEMA  = "default"
VOLUME_NAME = "bank_fraud"

BASE = f"/Volumes/preethiworkspace_7405608993331015/default/bank_fraud"

paths = {
  "landing":      f"{BASE}/landing/transactions",
  "schemaLoc":    f"{BASE}/schema/transactions",
  "ckpt_bronze":  f"{BASE}/checkpoints/bronze",
  "ckpt_silver":  f"{BASE}/checkpoints/silver",
  "ckpt_gold":    f"{BASE}/checkpoints/gold",
  "reference":    f"{BASE}/reference"
}

tables = {
  "bronze": f"{UC_CATALOG}.{UC_SCHEMA}.bronze_transactions",
  "silver": f"{UC_CATALOG}.{UC_SCHEMA}.silver_transactions",
  "gold":   f"{UC_CATALOG}.{UC_SCHEMA}.gold_alerts",
  "stats":  f"{UC_CATALOG}.{UC_SCHEMA}.dim_category_stats"
}

# quick sanity check
display(dbutils.fs.ls(BASE))
paths, tables

In [0]:
import pyspark.sql.functions as F

test_df = (spark.read
           .option("header", True)
           .option("inferSchema", True)
           .csv(f"{paths['reference']}/fraudTest.csv"))

# drop common "Unnamed" index col if present
cols = [c for c in test_df.columns if not c.lower().startswith("unnamed")]
test_df = test_df.select(cols)

# helpful trace column
test_df = test_df.withColumn("_seed_write_ts", F.current_timestamp())

n_chunks = 30
test_df = test_df.withColumn("_chunk", (F.monotonically_increasing_id() % n_chunks).cast("int"))

for i in range(n_chunks):
    chunk_dir = f"{paths['landing']}/chunk_{i:03d}"
    (test_df.filter(F.col("_chunk") == i)
           .drop("_chunk")
           .coalesce(1)
           .write.mode("overwrite")
           .option("header", True)
           .csv(chunk_dir))

display(dbutils.fs.ls(paths["landing"]))